In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# API Proxy Client Usage

This notebook demonstrates how to interact with the API Management gateway using the `google-genai` client library.

## Install Dependencies

This cell installs the `google-genai` library, which is required to interact with Google's Gemini models.

In [ ]:
%pip install 'google-genai>=1.52.0'

## Set Variables

In [ ]:
import subprocess

GCP_PROJECT_ID = subprocess.run(['terraform', 'output', '-raw', 'gemini_project_id'], capture_output=True, text=True, check=True).stdout.strip()
GCP_LOCATION = "us-central1"
GATEWAY_URL = subprocess.run(['terraform', 'output', '-raw', 'api_management_gateway_url'], capture_output=True, text=True, check=True).stdout.strip()
API_KEY="<UPDATE_ME>"

# Initialize Client

In [ ]:
from google import genai
from google.genai import types
from IPython.display import display, Image, Markdown, Video

client = genai.Client(
    vertexai=True,
    http_options=types.HttpOptions(
        base_url=f"{GATEWAY_URL}/vertex-ai",
        base_url_resource_scope=types.ResourceScope.COLLECTION,
        headers={
            "Ocp-Apim-Subscription-Key": API_KEY
        }
    )
)

# Generate Content

In [ ]:
response = client.models.generate_content(
    model="gemini-3-pro-preview",
    contents="Tell me a joke about space."
)

print(response.text)

# Streaming Generate Content

In [ ]:
for chunk in client.models.generate_content_stream(
    model="gemini-2.5-flash", contents="Tell me a story in 300 words."
):
    print(chunk.text, end='')

# Nano Banana Image Generation

In [ ]:
response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents=("Generate an image of the Eiffel tower with fireworks in the background."),
    config=types.GenerateContentConfig(
        response_modalities=[types.Modality.TEXT, types.Modality.IMAGE],
        candidate_count=1,
    ),
)
for part in response.candidates[0].content.parts:
    if part.text:
        display(Markdown(part.text))
    elif part.inline_data:
        display(Image(data=part.inline_data.data, width=500))

# Predict (Imagen generation)

In [ ]:
response = client.models.generate_images(
    model='imagen-4.0-fast-generate-001',
    prompt='An umbrella in the foreground, and a rainy night sky in the background',
    config=types.GenerateImagesConfig(
        number_of_images=1,
        include_rai_reason=True,
        output_mime_type='image/jpeg',
    ),
)
display(Image(data=response.generated_images[0].image.image_bytes, width=500))

# Long Running Predict (Veo)

In [ ]:
import time

# Create operation
operation = client.models.generate_videos(
    model='veo-3.1-fast-generate-001',
    prompt='a close-up shot of a golden retriever playing in a field of sunflowers',
    config=types.GenerateVideosConfig(
        number_of_videos=1,
        duration_seconds=8,
        enhance_prompt=True,
    ),
)

print(operation)

# Poll operation
while not operation.done:
    time.sleep(20)
    operation = client.operations.get(operation)
    print(operation)

generated_video = operation.result.generated_videos[0]

with open("video.mp4", "wb") as out_file:
    out_file.write(generated_video.video.video_bytes)
display(Video("video.mp4", embed=True, width=600, height=400))